In [ ]:
# Install & import dependencies (guarded)
import importlib, subprocess, sys

# Standard imports
import os
import sys
import json
import yaml
import logging
from pathlib import Path
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm import tqdm
import pandas as pd
# Project imports
from dataset import FacesDataset
from augmentations import get_train_transform, get_test_transform
from train_gan_ebm import train_gan_ebm_full
from gan_generate import generate_samples, show_images, save_samples, compute_fid, compute_is


In [2]:
# Device and cuDNN setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# default cudnn settings (can be overridden in config)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# expose device for downstream cells
device

Device: cuda


device(type='cuda')

In [3]:
# Global configuration dataclass and helpers
from dataclasses import dataclass, asdict

@dataclass
class Config:
    experiment_name: str = "gan_ebm_exp"
    seed: int = 42
    batch_size: int = 32
    lr_g: float = 2e-4
    lr_d: float = 2e-4
    lr_ebm: float = 1e-4
    betas: tuple = (0.5, 0.999)
    epochs: int = 20
    image_size: int = 64
    nz: int = 128
    g_channels: list = (512, 256, 128, 64)
    d_channels: list = (64, 128, 256, 512)
    ebm_steps: int = 5
    gp_lambda: float = 10.0
    dataset: str = "faces"
    dataroot: str = "../data/processed_64"
    num_workers: int = 4
    checkpoint_freq: int = 5
    deterministic: bool = False
    output_dir: str = "../outputs/gan_ebm"

    def save(self, path="gan_ebm_config.yaml"):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            yaml.safe_dump(asdict(self), f)

cfg = Config()
print(asdict(cfg))

{'experiment_name': 'gan_ebm_exp', 'seed': 42, 'batch_size': 32, 'lr_g': 0.0002, 'lr_d': 0.0002, 'lr_ebm': 0.0001, 'betas': (0.5, 0.999), 'epochs': 20, 'image_size': 64, 'nz': 128, 'g_channels': (512, 256, 128, 64), 'd_channels': (64, 128, 256, 512), 'ebm_steps': 5, 'gp_lambda': 10.0, 'dataset': 'faces', 'dataroot': '../data/processed_64', 'num_workers': 4, 'checkpoint_freq': 5, 'deterministic': False, 'output_dir': '../outputs/gan_ebm'}


In [4]:
# Load external config if present
for candidate in ("config.yaml", "dan_config.yaml", "gan_ebm_config.yaml"):
    if Path(candidate).exists():
        print("Loading config from", candidate)
        with open(candidate) as f:
            data = yaml.safe_load(f)
        # merge into cfg
        for k, v in data.items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)
        break

print("Effective config:")
print(asdict(cfg))

# Optionally override via argparse when run as script (left as comment for notebook)
# import argparse
# parser = argparse.ArgumentParser()
# parser.add_argument('--epochs', type=int)
# args = parser.parse_args()
# if args.epochs: cfg.epochs = args.epochs

Effective config:
{'experiment_name': 'gan_ebm_exp', 'seed': 42, 'batch_size': 32, 'lr_g': 0.0002, 'lr_d': 0.0002, 'lr_ebm': 0.0001, 'betas': (0.5, 0.999), 'epochs': 20, 'image_size': 64, 'nz': 128, 'g_channels': (512, 256, 128, 64), 'd_channels': (64, 128, 256, 512), 'ebm_steps': 5, 'gp_lambda': 10.0, 'dataset': 'faces', 'dataroot': '../data/processed_64', 'num_workers': 4, 'checkpoint_freq': 5, 'deterministic': False, 'output_dir': '../outputs/gan_ebm'}


In [5]:
# Paths, logging and experiment directory setup
output_dir = Path(cfg.output_dir)
ckpt_dir = output_dir / "checkpoints"
logs_dir = output_dir / "logs"
samples_dir = output_dir / "samples"
ckpt_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)
samples_dir.mkdir(parents=True, exist_ok=True)

# Logging
log_file = logs_dir / f"{cfg.experiment_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger()

# TensorBoard writer (if available)
try:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(logs_dir)
except Exception:
    writer = None

logger.info(f"Output dir: {output_dir}")
logger.info(f"Checkpoints: {ckpt_dir}")


2026-04-18 02:38:11,760 INFO Output dir: ../outputs/gan_ebm
2026-04-18 02:38:11,760 INFO Checkpoints: ../outputs/gan_ebm/checkpoints


In [6]:
# Random seeds and reproducibility
import torch
import numpy as np
import random

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

if cfg.deterministic:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# DataLoader worker init
def worker_init_fn(worker_id):
    seed = cfg.seed + worker_id
    np.random.seed(seed)
    random.seed(seed)

logger.info(f"Random seed set to {cfg.seed}")


2026-04-18 02:38:11,770 INFO Random seed set to 42


In [7]:
# Dataset and DataLoader stubs
from torch.utils.data import DataLoader

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
for df in [train_df, test_df]:
    df["image_path"] = df["id"].apply(lambda x: f"../data/processed_64/face-{int(x)}.png")

train_loader = DataLoader(
    FacesDataset(train_df, transform=get_train_transform(cfg.image_size)),
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    worker_init_fn=worker_init_fn,
)

val_loader = DataLoader(
    FacesDataset(test_df, transform=get_test_transform(cfg.image_size)),
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
)

logger.info(f"Train loader with {len(train_loader.dataset)} samples")


2026-04-18 02:38:12,060 INFO Train loader with 4500 samples


/tmp/ipykernel_21687/1816671620.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["image_path"] = df["id"].apply(lambda x: f"../data/processed_64/face-{int(x)}.png")


In [8]:
# Model scaffolds and instantiation
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.energy_discriminator import EnergyDiscriminator

# weight init helper
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G = Generator(latent_dim=cfg.nz, channels=cfg.g_channels, use_batchnorm=True, activation='relu').to(device)
D = Discriminator(channels=cfg.d_channels, use_batchnorm=True).to(device)
E = EnergyDiscriminator(channels=cfg.d_channels, use_batchnorm=True).to(device)

G.apply(weights_init)
D.apply(weights_init)
E.apply(weights_init)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

logger.info(f"Generator params: {count_params(G):,}")
logger.info(f"Discriminator params: {count_params(D):,}")
logger.info(f"Energy params: {count_params(E):,}")


2026-04-18 02:38:12,283 INFO Generator params: 3,807,043
2026-04-18 02:38:12,284 INFO Discriminator params: 2,766,529
2026-04-18 02:38:12,284 INFO Energy params: 2,766,529


In [9]:
# Optimizers, schedulers and loss stubs
opt_G = optim.Adam(G.parameters(), lr=cfg.lr_g, betas=cfg.betas)
opt_D = optim.Adam(D.parameters(), lr=cfg.lr_d, betas=cfg.betas)
opt_E = optim.Adam(E.parameters(), lr=cfg.lr_ebm, betas=cfg.betas)

# schedulers (optional)
from torch.optim.lr_scheduler import StepLR
sched_G = StepLR(opt_G, step_size=10, gamma=0.5)
sched_E = StepLR(opt_E, step_size=10, gamma=0.5)

# losses
bce_loss = nn.BCEWithLogitsLoss()
# EBM-specific losses will be computed in the training script


In [10]:
# Checkpoint utilities
import shutil

def save_checkpoint(state, path):
    tmp = str(path) + ".tmp"
    torch.save(state, tmp)
    shutil.move(tmp, str(path))

def save_models(epoch, G, E, opt_G, opt_E, path=ckpt_dir):
    path = Path(path) / f"checkpoint_epoch_{epoch}.pt"
    save_checkpoint({
        'epoch': epoch,
        'G_state': G.state_dict(),
        'E_state': E.state_dict(),
        'opt_G': opt_G.state_dict(),
        'opt_E': opt_E.state_dict(),
    }, path)

def load_latest_checkpoint(path=ckpt_dir):
    files = sorted(Path(path).glob('checkpoint_epoch_*.pt'))
    if not files:
        return None
    latest = files[-1]
    return torch.load(latest)


In [11]:
# Training loop scaffold (calls into train_gan_ebm)
# This cell shows how to run a full training using the helper

# save config
cfg.save(Path(cfg.output_dir) / 'gan_ebm_config.yaml')

G, E, G_losses, E_losses = train_gan_ebm_full(
    config=asdict(cfg),
    device=device,
    train_loader=train_loader,
    epochs=cfg.epochs,
)

logger.info(f"Finished training. Last G loss: {G_losses[-1]:.4f}, Last E loss: {E_losses[-1]:.4f}")

# Save models
save_models(cfg.epochs-1, G, E, opt_G, opt_E, path=ckpt_dir)

# Generate and save samples
save_samples(G, device, str(samples_dir / 'ebm_final.png'), latent_dim=cfg.nz)

# Evaluate
fid = compute_fid(G, val_loader, device, latent_dim=cfg.nz)
is_score = compute_is(G, device, latent_dim=cfg.nz)
print(f"FID: {fid:.2f}, IS: {is_score:.2f}")


Epoch [1/20]:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch [2/20]:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch [3/20]:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch [4/20]:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch [5/20]:   0%|          | 0/141 [00:00<?, ?it/s]

...

Epoch [19/20]:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch [20/20]:   0%|          | 0/141 [00:00<?, ?it/s]

2026-04-18 02:45:19,675 INFO Finished training. Last G loss: 1508.4609, Last E loss: -19.1016


/home/crisp/Projects/Gen AI - A1/.venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


FID: 305.48, IS: 2.31
